# SNNAI v6-6-1 Bio-NAS LM Architecture Search Demo

Demonstrates Phase 7.0 Bio-NAS for language model architecture search.
- Uses small proxy LM models (embed_dim=32, hidden_dim=64, seq_len=32)
- Population 6, generations 3 (fast for Kaggle)
- Multi-objective: perplexity, latency, energy, BLEU-1, biological penalty
- Reports best architecture and Pareto front

In [ ]:
# Install and clone
!pip install -q torch numpy
!rm -rf snnai
!git clone --depth 1 --branch v6.6.0 https://github.com/sabunosuke1008-create/snnai.git
import sys
sys.path.insert(0, 'snnai')

In [ ]:
# Run Bio-NAS LM search
from snnai.bio_nas import search_lm_architecture, lm_serial_architecture, lm_diverse_architecture

print("Starting Bio-NAS LM search (Phase 7.0)...")
print("Small proxy: vocab=128, embed=32, hidden=64, seq=32, epochs=2")
print()

eval_kwargs = {
    "vocab_size": 128,
    "embed_dim": 32,
    "hidden_dim": 64,
    "seq_len": 32,
    "epochs": 2,
}

best_arch, best_metrics, history, pareto = search_lm_architecture(
    population_size=6,
    n_generations=3,
    top_k=3,
    seed=42,
    eval_kwargs=eval_kwargs,
)

print("=" * 60)
print("BEST ARCHITECTURE")
print("=" * 60)
print("Nodes:", best_arch.nodes)
print("Edges:", best_arch.edges)
print("Layer types used:", best_arch.count_lm_layer_types())
print()
print("=" * 60)
print("BEST METRICS")
print("=" * 60)
for k, v in best_metrics.items():
    print(f"  {k}: {v}")
print()
print("=" * 60)
print("SEARCH HISTORY (best score per generation)")
print("=" * 60)
for gen, score, metrics in history:
    print(f"  Gen {gen}: score={score:.4f}, val_ppl={metrics['val_ppl']:.2f}")
print()
print("=" * 60)
print(f"PARETO FRONT ({len(pareto)} non-dominated architectures)")
print("=" * 60)
for i, (arch, metrics) in enumerate(pareto):
    print(f"  [{i}] layers={list(arch.count_lm_layer_types())}, ppl={metrics['val_ppl']:.2f}, latency={metrics['latency_sec']:.4f}s")

In [ ]:
# Save results
import json
results = {
    'best_arch_nodes': best_arch.nodes,
    'best_arch_edges': [list(e) for e in best_arch.edges],
    'best_arch_layer_types': {n: best_arch.layers[n].layer_type for n in best_arch.layers},
    'best_metrics': {k: float(v) if hasattr(v, 'item') else v for k, v in best_metrics.items()},
    'history': [(gen, float(score), {k: float(vv) if hasattr(vv, 'item') else vv for k, vv in metrics.items()}) for gen, score, metrics in history],
    'pareto_count': len(pareto),
}
with open('/kaggle/working/v6_6_0_bionas_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved /kaggle/working/v6_6_0_bionas_results.json')

## Summary

This notebook demonstrates Bio-NAS Phase 7.0: evolutionary search over LM layer architectures
(feedforward, recurrent, attention, hippocampus_gate) with multi-objective optimization.
The best architecture found is reported along with a Pareto front of non-dominated solutions.